# Busca Adversarial de Transformações — quais métodos e quanto de cada

**Ideia central:** um classificador que identifica **qual gerador** fez cada imagem (9 classes: 8 geradores do ArtiFact + o **StyleGAN do 140k**). Uma boa transformação é a que deixa esse classificador **o mais confuso possível** — sinal de que o resíduo específico de fonte foi destruído — **sem destruir a imagem**.

**Decisões de design (e por quê):**
1. **Adversário re-treinado por candidato** — um classificador fixo pode ser enganado escondendo só as features dele; o honesto é re-treinar sobre as imagens transformadas e ver se *ainda assim* ele distingue os geradores. Probe linear sobre features baratas → re-treino em segundos.
2. **4 sondas de medição**: espectro radial + **picos azimutais** (novo — pega grades 2D, torna o `whiten` julgável) + cor + resíduo local.
3. **Fitness com restrição**: `destruição = (acc_base − acc_t) / (acc_base − chance)` (1.0 = adversário reduzido à chance), válida só se **SSIM ≥ piso** (senão é destruição da imagem, não do fingerprint).
4. **Busca em 2 estágios**: (A) varredura fina por método → valor ótimo de cada; (B) encadeamento guloso dos top-4 → receita final.

**Anti-vazamento:** usa a metade **dev** do ArtiFact (a `test` fica intocada para a confirmação com CNN no notebook 02).

**Saída automática:** ranking completo + `best_recipe.json` com a receita (métodos + valores exatos) e faixas sugeridas para `RandomAugment`.

> ~70 candidatos x ~20-30s ≈ 30-45 min. Salva incrementalmente — pode interromper e retomar.

In [ ]:
import sys, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score

sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import apply_preprocess, artifact_split

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"
ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
RAW_140K     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
RESULTS_DIR  = PROJECT_ROOT / "artifacts" / "adversarial_search"
FIGS_DIR     = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); FIGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "search_results.json"

IMG_SIZE       = 256
N_PER_CLASS    = 70      # imagens por classe (9 classes ~ 630 imagens)
N_REAL_DIAG    = 150     # reais so para o diagnostico final
SSIM_PER_CLASS = 8       # pares por classe para medir SSIM
SSIM_FLOOR     = 0.70    # piso de integridade da imagem
SEED           = 42
rng = np.random.default_rng(SEED)

print("DATA_ROOT:", DATA_ROOT)
print("ArtiFact:", ARTIFACT_DIR.exists(), "| 140k:", RAW_140K.exists())

## 1. Dados — 9 classes de gerador (metade dev) + reais para diagnóstico

As 9 classes: 8 geradores do ArtiFact + `stylegan_140k` (fakes do dataset de treino). Incluir o StyleGAN importa: é **dele** que o modelo real aprende os atalhos — o adversário precisa enxergá-lo.

In [ ]:
def list_by_source(folder):
    g = {}
    for p in folder.glob("*.*"):
        s = p.name.split("__")[0] if "__" in p.name else "?"
        g.setdefault(s, []).append(p)
    return {k: sorted(v) for k, v in g.items()}

def load_rgb(path):
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

def pick(files, n, seed=SEED):
    files = list(files)
    if n >= len(files):
        return files
    r = np.random.default_rng(seed)
    return [files[i] for i in r.choice(len(files), n, replace=False)]

fake_groups = list_by_source(ARTIFACT_DIR / "fake")
CLASSES = sorted(fake_groups) + ["stylegan_140k"]
CHANCE = 1.0 / len(CLASSES)

class_pils = {}
for g in sorted(fake_groups):
    dev = artifact_split(fake_groups[g], which="dev")     # metade de SELECAO
    class_pils[g] = [load_rgb(p) for p in pick(dev, N_PER_CLASS)]

sty_files = sorted((RAW_140K / "train" / "fake").glob("*.jpg"))
class_pils["stylegan_140k"] = [load_rgb(p) for p in pick(sty_files, N_PER_CLASS)]

real_groups = list_by_source(ARTIFACT_DIR / "real")
real_pils = []
for s in sorted(real_groups):
    dev = artifact_split(real_groups[s], which="dev")
    real_pils += [load_rgb(p) for p in pick(dev, max(1, N_REAL_DIAG // len(real_groups)))]

print("classes:", CLASSES)
print("imagens/classe:", {c: len(class_pils[c]) for c in CLASSES})
print("reais (diagnostico):", len(real_pils), "| chance 9-classes:", round(CHANCE, 3))

## 2. As 4 sondas + o adversário

Feature por imagem = concat( espectro radial, **picos azimutais**, cor, resíduo local ). Adversário = regressão logística multiclasse, avaliada por **acurácia balanceada** com CV estratificado — re-treinada do zero a cada candidato.

In [ ]:
# mapa de raios precomputado (reusado em todas as FFTs)
_yy, _xx = np.indices((IMG_SIZE, IMG_SIZE))
R = np.hypot(_xx - IMG_SIZE // 2, _yy - IMG_SIZE // 2).astype(int)
RMAX = int(R.max()) + 1
NR = np.maximum(np.bincount(R.ravel(), minlength=RMAX), 1)
NFREQ = IMG_SIZE // 2

def _box3(a):
    p = np.pad(a, 1, mode="reflect")
    return (p[:-2, :-2] + p[:-2, 1:-1] + p[:-2, 2:] +
            p[1:-1, :-2] + p[1:-1, 1:-1] + p[1:-1, 2:] +
            p[2:, :-2] + p[2:, 1:-1] + p[2:, 2:]) / 9.0

def feats_all(pil):
    rgb  = np.asarray(pil.convert("RGB"), dtype=np.float32) / 255.0
    gray = np.asarray(pil.convert("L"), dtype=np.float32) / 255.0
    # --- espectro: perfil radial + picos azimutais (std do log-mag por anel) ---
    F = np.fft.fftshift(np.abs(np.fft.fft2(gray)))
    logmag = np.log1p(F)
    radial = np.log1p(np.bincount(R.ravel(), (F ** 2).ravel(), minlength=RMAX) / NR)[:NFREQ]
    m1 = np.bincount(R.ravel(), logmag.ravel(), minlength=RMAX) / NR
    m2 = np.bincount(R.ravel(), (logmag ** 2).ravel(), minlength=RMAX) / NR
    peaks = np.sqrt(np.maximum(m2 - m1 ** 2, 0))[:NFREQ]
    # --- cor ---
    col = []
    for c in range(3):
        ch = rgb[:, :, c].ravel()
        m, s = ch.mean(), ch.std() + 1e-6
        col += [m, s, float(((ch - m) ** 3).mean() / s ** 3), float(((ch - m) ** 4).mean() / s ** 4)]
    Rc, Gc, Bc = rgb[:, :, 0].ravel(), rgb[:, :, 1].ravel(), rgb[:, :, 2].ravel()
    col += [np.corrcoef(Rc, Gc)[0, 1], np.corrcoef(Rc, Bc)[0, 1], np.corrcoef(Gc, Bc)[0, 1]]
    mx, mn = rgb.max(2), rgb.min(2)
    sat = (mx - mn) / (mx + 1e-6)
    col += [sat.mean(), sat.std()]
    for c in range(3):
        h, _ = np.histogram(rgb[:, :, c], bins=8, range=(0, 1), density=True)
        col += h.tolist()
    # --- residuo local (co-ocorrencia passa-alta) ---
    g255 = gray * 255.0
    resid = g255 - _box3(g255)
    q = (np.clip(np.round(resid), -3, 3) + 3).astype(int)
    hh, vv = q[:, :-1].ravel(), q[:, 1:].ravel()
    co = np.zeros((7, 7)); np.add.at(co, (hh, vv), 1); co /= max(co.sum(), 1)
    hist, _ = np.histogram(resid, bins=16, range=(-8, 8), density=True)
    res = np.concatenate([co.ravel(), hist, [resid.std()]])
    return np.nan_to_num(np.concatenate([radial, peaks, np.array(col), res]).astype(np.float32))

def balanced_acc(X, y):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
    return float(cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy").mean())

def ssim_gray(pa, pb, sigma=1.5):
    a = np.asarray(pa.convert("L"), dtype=np.float32) / 255.0
    b = np.asarray(pb.convert("L"), dtype=np.float32) / 255.0
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ma, mb = gaussian_filter(a, sigma), gaussian_filter(b, sigma)
    va = gaussian_filter(a * a, sigma) - ma ** 2
    vb = gaussian_filter(b * b, sigma) - mb ** 2
    cov = gaussian_filter(a * b, sigma) - ma * mb
    s = ((2 * ma * mb + C1) * (2 * cov + C2)) / ((ma ** 2 + mb ** 2 + C1) * (va + vb + C2))
    return float(s.mean())

print("Sondas e adversario definidos. dim(features) =", len(feats_all(class_pils[CLASSES[0]][0])))

## 3. Baseline — quão identificável é cada gerador?

Acurácia balanceada do adversário sobre as imagens **limpas** + matriz de confusão. Esse número é o que vamos tentar derrubar até a chance (~0.11).

In [ ]:
def dataset_feats(chain):
    # aplica a cadeia de transformacoes e extrai features de todas as classes
    np.random.seed(0)   # transforms estocasticos (noise) reprodutiveis
    X, y = [], []
    for ci, cls in enumerate(CLASSES):
        for pil in class_pils[cls]:
            t = pil
            for m, v in chain:
                t = apply_preprocess(t, m, v)
            X.append(feats_all(t)); y.append(ci)
    return np.stack(X), np.array(y)

def eval_candidate(chain):
    t0 = time.time()
    X, y = dataset_feats(chain)
    acc = balanced_acc(X, y)
    np.random.seed(0)
    ssims = []
    for cls in CLASSES:
        for pil in class_pils[cls][:SSIM_PER_CLASS]:
            t = pil
            for m, v in chain:
                t = apply_preprocess(t, m, v)
            ssims.append(ssim_gray(pil, t))
    return acc, float(np.mean(ssims)), time.time() - t0

X0, y0 = dataset_feats([])
ACC_BASE = balanced_acc(X0, y0)
print(f"Adversario (9 classes, imagens limpas): acc balanceada = {ACC_BASE:.3f}  (chance = {CHANCE:.3f})")

# matriz de confusao baseline
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pred0 = cross_val_predict(clf, X0, y0, cv=StratifiedKFold(3, shuffle=True, random_state=SEED))
cm0 = confusion_matrix(y0, pred0, normalize="true")
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm0, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=7)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=7)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cm0[i, j] > 0.02:
            ax.text(j, i, f"{cm0[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm0[i, j] > 0.5 else "black")
ax.set_title(f"Adversario em imagens LIMPAS (acc bal. {ACC_BASE:.2f}) — diagonal forte = fingerprints fortes")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGS_DIR / "adv_confusao_baseline.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Estágio A — varredura fina por método

Para cada método x valor: re-treina o adversário nas imagens transformadas. **`destruicao`** = fração do fingerprint destruída (1.0 = adversário na chance). Válido só se `ssim >= piso`. Salva incrementalmente (retoma se interromper).

In [ ]:
GRIDS = {   # ordenados do mais leve ao mais forte
    "jpeg":        [90, 80, 70, 60, 50, 40, 30],
    "blur":        [0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0],
    "downscale":   [1.25, 1.5, 1.75, 2.0, 2.5, 3.0],
    "noise":       [0.02, 0.04, 0.06, 0.08, 0.10, 0.12],
    "whiten":      [0.3, 0.5, 0.8, 1.0],
    "posterize":   [6, 5, 4, 3],
    "gamma":       [0.8, 1.3, 0.6, 1.6],
    "saturation":  [0.6, 1.5, 0.3, 2.0],
    "grayscale":   [0.5, 1.0],
    "histeq":      [1],
    "chanshuffle": [1],
}
IDENTITY = {"jpeg": 100, "blur": 0.0, "downscale": 1.0, "noise": 0.0, "whiten": 0.0,
            "posterize": 8, "gamma": 1.0, "saturation": 1.0, "grayscale": 0.0}

def chain_id(chain):
    return " > ".join(f"{m}:{v}" for m, v in chain) if chain else "(limpa)"

if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    done = {r["id"] for r in results}
    print(f"Retomando: {len(results)} candidatos avaliados.")
else:
    results, done = [], set()

def record(chain):
    cid = chain_id(chain)
    if cid in done:
        return
    acc, ssim, dt = eval_candidate(chain)
    destr = (ACC_BASE - acc) / (ACC_BASE - CHANCE)
    results.append({"id": cid, "chain": [[m, v] for m, v in chain],
                    "acc": round(acc, 4), "destruicao": round(destr, 4),
                    "ssim": round(ssim, 4), "valido": bool(ssim >= SSIM_FLOOR), "t": round(dt, 1)})
    done.add(cid)
    RESULTS_PATH.write_text(json.dumps(results, indent=1))
    flag = "ok " if ssim >= SSIM_FLOOR else "X  "
    print(f" {flag}{cid:28s} acc={acc:.3f} destruicao={destr:+.2f} ssim={ssim:.3f} ({dt:.0f}s)")

n_total = sum(len(v) for v in GRIDS.values())
print(f"Estagio A: {n_total} candidatos\n")
for method, values in GRIDS.items():
    for v in values:
        record([(method, v)])
print("\nEstagio A concluido.")

## 5. Estágio B — encadeamento guloso (top-4)

Toma os 4 métodos válidos com maior destruição (no melhor valor de cada um) e testa **pares ordenados** (a→b e b→a; a ordem importa — jpeg por último imita pipeline real), cada par também em **variante leve** (valores a meio caminho da identidade — empilhar métodos compõe a degradação e pode furar o piso de SSIM). Depois tenta estender o melhor par para **trinca**.

In [ ]:
def mild(method, value):
    idv = IDENTITY.get(method)
    if idv is None:
        return None
    mv = (value + idv) / 2
    if method in ("jpeg", "posterize"):
        return int(round(mv))
    return round(mv, 3)

valid_singles = [r for r in results if len(r["chain"]) == 1 and r["valido"]]
best_by_method = {}
for r in valid_singles:
    m = r["chain"][0][0]
    if m not in best_by_method or r["destruicao"] > best_by_method[m]["destruicao"]:
        best_by_method[m] = r
top4 = sorted(best_by_method.values(), key=lambda r: -r["destruicao"])[:4]
print("Top-4 metodos (no melhor valor valido):")
for r in top4:
    print(f"  {r['id']:20s} destruicao={r['destruicao']:.2f} ssim={r['ssim']:.3f}")

# pares ordenados (a -> b), versao otima e versao leve
print("\nPares ordenados:")
for ra in top4:
    for rb in top4:
        ma, va = ra["chain"][0]; mb, vb = rb["chain"][0]
        if ma == mb:
            continue
        record([(ma, va), (mb, vb)])
        la, lb = mild(ma, va), mild(mb, vb)
        if la is not None and lb is not None:
            record([(ma, la), (mb, lb)])

# melhor par valido -> tenta estender para trinca (com valores leves)
pares = [r for r in results if len(r["chain"]) == 2 and r["valido"]]
if pares:
    best_pair = max(pares, key=lambda r: r["destruicao"])
    print(f"\nMelhor par: {best_pair['id']} (destruicao {best_pair['destruicao']:.2f}) — tentando trincas:")
    usados = {m for m, _ in best_pair["chain"]}
    for r in top4:
        m, v = r["chain"][0]
        if m in usados:
            continue
        lv = mild(m, v)
        record([tuple(c) for c in best_pair["chain"]] + [(m, lv if lv is not None else v)])
print("\nEstagio B concluido.")

## 6. Resultado — a receita: quais métodos e quanto de cada

Ranking dos candidatos válidos (SSIM ≥ piso) por destruição do fingerprint. Salva `best_recipe.json` com a cadeia vencedora + faixas sugeridas para `RandomAugment` (de "metade do caminho até a identidade" ao valor ótimo).

In [ ]:
df = pd.DataFrame(results)
ok = df[df.valido].sort_values("destruicao", ascending=False)
print(f"Baseline adversario: {ACC_BASE:.3f} | chance: {CHANCE:.3f} | piso SSIM: {SSIM_FLOOR}\n")
print("TOP-15 candidatos validos:\n")
print(ok.head(15)[["id", "acc", "destruicao", "ssim"]].to_string(index=False))

best = ok.iloc[0]
best_chain = [tuple(c) for c in best["chain"]]
print(f"\n=> RECEITA VENCEDORA: {best['id']}")
print(f"   adversario {ACC_BASE:.3f} -> {best['acc']:.3f} (destruicao {best['destruicao']:.2f}) | ssim {best['ssim']:.3f}")

pool_ranges = {}
for m, v in best_chain:
    lv = mild(m, v)
    pool_ranges[m] = sorted([lv if lv is not None else v, v])

recipe = {
    "chain": [[m, v] for m, v in best_chain],
    "adversary_acc_clean": round(ACC_BASE, 4), "adversary_acc_after": float(best["acc"]),
    "destruicao": float(best["destruicao"]), "ssim": float(best["ssim"]),
    "chance": round(CHANCE, 4), "ssim_floor": SSIM_FLOOR,
    "suggested_pool_ranges": pool_ranges,
    "nota": "selecionado na metade dev do ArtiFact; confirmar com CNN no notebook 02 (metade test)",
}
(RESULTS_DIR / "best_recipe.json").write_text(json.dumps(recipe, indent=2))
print("\nFaixas sugeridas p/ RandomAugment:", pool_ranges)
print("Salvo em:", RESULTS_DIR / "best_recipe.json")

# scatter destruicao x ssim (todos os candidatos)
fig, ax = plt.subplots(figsize=(8.5, 5.5))
meths = sorted({r["chain"][0][0] if len(r["chain"]) == 1 else "cadeia" for r in results})
cmap = plt.cm.tab20(np.linspace(0, 1, len(meths)))
for mt, cor in zip(meths, cmap):
    sel = [r for r in results if (r["chain"][0][0] if len(r["chain"]) == 1 else "cadeia") == mt]
    ax.scatter([r["ssim"] for r in sel], [r["destruicao"] for r in sel], s=28, color=cor, label=mt,
               marker="s" if mt == "cadeia" else "o")
ax.axvline(SSIM_FLOOR, color="red", ls="--", lw=1, label=f"piso SSIM ({SSIM_FLOOR})")
ax.axhline(1.0, color="green", ls=":", lw=1, label="chance (destruicao total)")
ax.set_xlabel("SSIM (integridade da imagem)"); ax.set_ylabel("destruicao do fingerprint")
ax.set_title("Cada ponto = uma transformacao; canto superior direito = ideal")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS_DIR / "adv_scatter.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# matriz de confusao DEPOIS da receita vencedora — quem ficou indistinguivel de quem?
Xb, yb = dataset_feats(best_chain)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
predb = cross_val_predict(clf, Xb, yb, cv=StratifiedKFold(3, shuffle=True, random_state=SEED))
cmb = confusion_matrix(yb, predb, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, cm, ttl in [(axes[0], cm0, f"ANTES (acc {ACC_BASE:.2f})"),
                    (axes[1], cmb, f"DEPOIS da receita (acc {best['acc']:.2f})")]:
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=7)
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=7)
    ax.set_title(ttl)
plt.suptitle("Adversario de geradores: confusao antes vs depois")
plt.tight_layout()
plt.savefig(FIGS_DIR / "adv_confusao_antes_depois.png", dpi=130, bbox_inches="tight")
plt.show()

## 7. Diagnóstico — destruímos o atalho sem cegar o real-vs-fake?

Para o top-3, mede a separabilidade **real-vs-fake** (probe binário, reais e fakes ambos transformados). Se ela despencar para ~0.5 junto com a confusão de geradores, a transformação está destruindo *todo* o sinal — não só o fingerprint. O ideal: confusão de geradores alta, real-vs-fake ainda acima da chance.

In [ ]:
def realfake_auc(chain):
    np.random.seed(0)
    def tf(pil):
        t = pil
        for m, v in chain:
            t = apply_preprocess(t, m, v)
        return t
    fakes = []
    for cls in CLASSES:
        fakes += [tf(p) for p in class_pils[cls][:20]]
    reais = [tf(p) for p in real_pils]
    X = np.stack([feats_all(p) for p in reais] + [feats_all(p) for p in fakes])
    y = np.r_[np.zeros(len(reais)), np.ones(len(fakes))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return float(cross_val_score(clf, X, y, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                                 scoring="roc_auc").mean())

auc_clean = realfake_auc([])
print(f"real-vs-fake (probe, imagens limpas): AUC = {auc_clean:.3f}\n")
print(f"{'candidato':30s} {'destr.':>7} {'ssim':>6} {'real-vs-fake':>13}")
for _, r in ok.head(3).iterrows():
    auc_t = realfake_auc([tuple(c) for c in r["chain"]])
    print(f"{r['id']:30s} {r['destruicao']:7.2f} {r['ssim']:6.3f} {auc_t:13.3f}")
print("\nLeitura: queremos destruicao alta COM real-vs-fake ainda > 0.5 — o sinal compartilhado sobrevive.")

## 8. Leitura e próximos passos

- **A resposta a "quais métodos e quanto"** está em `best_recipe.json`: a cadeia vencedora com valores exatos + faixas sugeridas para treino (`suggested_pool_ranges`).
- A **matriz de confusão antes/depois** mostra *quem* ficou indistinguível de quem — geradores que colapsam juntos perderam o fingerprint que os separava.
- O **diagnóstico real-vs-fake** protege contra a vitória trivial (destruir tudo): a receita boa confunde geradores e *mantém* sinal real-vs-fake.

**Caveats honestos:**
- O adversário é um probe linear sobre features projetadas — um CNN adversário poderia achar resíduos que ele não vê (risco de Goodhart). Por isso a receita é **hipótese**, não conclusão.
- Seleção feita na metade **dev**; a confirmação obrigatória é treinar o CNN com a receita (notebook 02, `RECIPES`) e medir na metade **test**.
- No treino real a transformação entra com `p_apply < 1` e valores sorteados na faixa — o modelo também vê imagens limpas.

**Próximo passo:** adicionar a receita vencedora ao `RECIPES` do notebook 02 e rodar a confirmação com CNN.